# 03 - Daily scoring: risk-ranked coordinator worklist with GenAI briefs

This runs every morning after the Lakeflow pipeline:
1. Loads `lakeview_ml.readmission_risk@champion` from Unity Catalog.
2. Scores every index discharge. Historical scores go to `patient_risk_scores` (used for monitoring and Genie). Patients still inside their 30-day window become the **worklist**.
3. Explains each score: the **top 3 risk drivers** come from occlusion attribution (each driver is reset to a low-risk reference value, and we measure how much the risk drops).
4. Writes a **coordinator brief** for High and Medium risk patients with `ai_query` (a Databricks-hosted Claude model, so PHI stays in the platform).
5. Publishes `lakeview_ml.coordinator_worklist` (Change Data Feed on). This is the table that is synced to Lakebase for the app.

In [0]:
import mlflow, numpy as np, pandas as pd
from mlflow import MlflowClient

CAT = "simo_show_case_catalog"
MODEL = f"{CAT}.lakeview_ml.readmission_risk"
mlflow.set_registry_uri("databricks-uc")
model_version = MlflowClient(registry_uri="databricks-uc").get_model_version_by_alias(MODEL, "champion").version
model = mlflow.sklearn.load_model(f"models:/{MODEL}/{model_version}")
FEATURES = list(model.feature_names_in_)

df = spark.sql(f"""
  SELECT f.*, s.sdoh_lives_alone, s.sdoh_caregiver_available, s.sdoh_transportation_barrier, s.sdoh_medication_cost_barrier,
         s.sdoh_health_literacy_concern, s.sdoh_substance_use, s.social_risk_summary
  FROM {CAT}.lakeview_health.gold_discharge_features f
  JOIN {CAT}.lakeview_ml.note_social_risk s USING (encounter_id)
""").toPandas()
for c in FEATURES:
    if df[c].dtype == bool:
        df[c] = df[c].astype(int)

df["risk_score"] = model.predict_proba(df[FEATURES])[:, 1]
hist_scores = np.sort(df.loc[df.label_window_complete, "risk_score"].values)
df["risk_percentile"] = (np.searchsorted(hist_scores, df.risk_score, side="right") / len(hist_scores) * 100).round(1)
df["risk_tier"] = np.where(df.risk_percentile >= 80, "High", np.where(df.risk_percentile >= 60, "Medium", "Low"))
print(f"Scored {len(df):,} discharges with {MODEL} v{model_version}")
print("Worklist pool (30-day window still open):", int((~df.label_window_complete).sum()))
print(df[~df.label_window_complete].risk_tier.value_counts().to_dict())

{"ts": "2026-11-04 06:04:56.561", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzkyNzc3NzQwODkzMDM3MhABIAEyJDAxYTFhNTI4LWNlYmMtN2IzYi04Mjk0LTYyM2I5NzNiM2RhOTokMGVhMmU4M2MtMGNkMy0zODA0LThlYTktZmYyYzhiZmJmN2ZmSgwI9ZWr1wYQgKrzzwFQAVgBYAFokp6FgfDEow2IAQA=.", "context": {}}
{"ts": "2026-11-04 06:04:56.561", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzkyNzc3NzQwODkzMDM3MhABIAEyJDAxYTFhNTI4LWNlYmMtN2IzYi04Mjk0LTYyM2I5NzNiM2RhOTokMGVhMmU4M2MtMGNkMy0zODA0LThlYTktZmYyYzhiZmJmN2ZmSgwI9ZWr1wYQgKrzzwFQAVgBYAFokp6FgfDEow2IAQA=.", "context": {}}
{"ts": "2026-11-04 06:04:56.561", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzkyNzc3NzQwODkzMDM3MhABIAEyJDAxYTFhNTI4LWNlYmMtN2IzYi04Mjk0LTYyM2I5NzNiM2RhOTokMGVhMmU4M2MtMGNkMy0zODA0LThlYTktZmYyYzhi

Scored 12,989 discharges with simo_show_case_catalog.lakeview_ml.readmission_risk v1
Worklist pool (30-day window still open): 254
{'Low': 123, 'High': 71, 'Medium': 60}


In [0]:
hist = df[df.label_window_complete]
med = lambda c: float(hist[c].median())
# driver label -> low-risk reference values to substitute
DRIVERS = {
    "Lives alone":                         {"sdoh_lives_alone": 0},
    "No caregiver at home":                {"sdoh_caregiver_available": 1},
    "No reliable transportation":          {"sdoh_transportation_barrier": 0},
    "Cannot afford medications":           {"sdoh_medication_cost_barrier": 0},
    "Health literacy / language barrier":  {"sdoh_health_literacy_concern": 0},
    "Substance use":                       {"sdoh_substance_use": 0},
    "Frequent ED visits (6 mo)":           {"ed_visits_prior_6m": 0},
    "Prior admissions (12 mo)":            {"prior_admissions_12m": 0},
    "Multiple comorbidities":              {"comorbidity_count": med("comorbidity_count")},
    "Long length of stay":                 {"los_days": med("los_days")},
    "Polypharmacy":                        {"num_discharge_meds": med("num_discharge_meds")},
    "Abnormal labs at discharge":          {"sodium": med("sodium"), "hemoglobin": med("hemoglobin"), "creatinine": med("creatinine"), "bnp": med("bnp"), "abnormal_lab_count": 0},
    "No follow-up appointment scheduled":  {"followup_scheduled_7d": 1},
    "High-risk discharge disposition":     {"discharge_disposition": "Home"},
}

wl = df[~df.label_window_complete].copy().reset_index(drop=True)
base = wl.risk_score.values
deltas = {}
for label, ref in DRIVERS.items():
    X = wl[FEATURES].copy()
    for col, val in ref.items():
        X[col] = val
    deltas[label] = base - model.predict_proba(X)[:, 1]
delta_df = pd.DataFrame(deltas)
wl["top_drivers"] = [
    " | ".join([lab for lab, d in row.sort_values(ascending=False).head(3).items() if d > 0.01]) or "No dominant driver"
    for _, row in delta_df.iterrows()]
display(wl[["encounter_id", "primary_condition", "risk_percentile", "risk_tier", "top_drivers", "social_risk_summary"]]
        .sort_values("risk_percentile", ascending=False).head(10))

encounter_id,primary_condition,risk_percentile,risk_tier,top_drivers,social_risk_summary
E0007925,Acute MI,99.9,High,Frequent ED visits (6 mo) | Multiple comorbidities | Long length of stay,Lives alone with no caregiver available
E0012227,Heart failure,99.8,High,Frequent ED visits (6 mo) | Multiple comorbidities | Long length of stay,No follow-up appointment scheduled; caregiver available only for first week
E0006834,Diabetes with complications,99.4,High,Frequent ED visits (6 mo) | Multiple comorbidities | Lives alone,"Living alone since spouse death, heavy daily alcohol use 4-6 drinks, no follow-up appointment scheduled at discharge, no caregiver support identified"
E0002969,GI bleed,99.2,High,Multiple comorbidities | Frequent ED visits (6 mo) | Polypharmacy,null
E0010041,Urinary tract infection,99.1,High,Frequent ED visits (6 mo) | Multiple comorbidities | Polypharmacy,"Lives alone with no caregiver support, heavy alcohol use (4-6 drinks/day), medication cost barrier"
E0006833,Heart failure,99.0,High,Frequent ED visits (6 mo) | Lives alone | Multiple comorbidities,Lives alone
E0001774,Heart failure,99.0,High,Frequent ED visits (6 mo) | Multiple comorbidities | Polypharmacy,Heavy alcohol use (4-6 drinks daily)
E0006222,COPD exacerbation,98.7,High,Frequent ED visits (6 mo) | Multiple comorbidities | No follow-up appointment scheduled,"Transportation difficulties, medication cost barriers, health literacy challenges, daily alcohol use."
E0008412,Heart failure,98.7,High,Long length of stay | Frequent ED visits (6 mo) | Multiple comorbidities,null
E0011383,Heart failure,98.5,High,Frequent ED visits (6 mo) | Lives alone | Abnormal labs at discharge,"Lives alone, current smoker (1 pack/day), no caregiver support, no follow-up appointment scheduled"


In [0]:
from pyspark.sql import functions as F
scores = df[["encounter_id", "patient_id", "facility_id", "discharge_date", "risk_score", "risk_percentile", "risk_tier", "label_window_complete", "readmitted_30d"]].copy()
scores["model_version"] = str(model_version)
(spark.createDataFrame(scores).withColumn("scored_at", F.current_timestamp())
   .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.lakeview_ml.patient_risk_scores"))
spark.sql(f"COMMENT ON TABLE {CAT}.lakeview_ml.patient_risk_scores IS 'Readmission risk score for every index discharge (model readmission_risk@champion). risk_tier: High = top 20% percentile vs historical discharges, Medium = 60-80th, Low = rest. Join to lakeview_health.gold_index_admissions on encounter_id.'")

wl_out = wl[["encounter_id", "risk_score", "risk_percentile", "risk_tier", "top_drivers", "social_risk_summary"]].copy()
wl_out["model_version"] = str(model_version)
spark.createDataFrame(wl_out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.lakeview_ml.worklist_scored")
print(spark.table(f"{CAT}.lakeview_ml.patient_risk_scores").count(), "scores |", spark.table(f"{CAT}.lakeview_ml.worklist_scored").count(), "worklist rows")

12989 scores | 254 worklist rows


In [0]:
%sql
-- Today's worklist (GenAI brief generated once, then materialized into two tables)
CREATE OR REPLACE TEMP VIEW worklist_today AS
WITH base AS (
  SELECT w.encounter_id, a.patient_id, p.mrn, p.first_name, p.last_name, p.dob, a.facility_id, a.discharge_ts,
         datediff(current_date(), a.discharge_date) AS days_since_discharge,
         a.primary_condition, a.hrrp_condition, a.discharge_disposition, a.age_at_admit, a.sex, a.payer, a.primary_language,
         f.lace_score, f.los_days, f.num_discharge_meds, f.followup_scheduled_7d, f.followup_provider_type,
         w.risk_score, w.risk_percentile, w.risk_tier, w.top_drivers, w.social_risk_summary, w.model_version
  FROM simo_show_case_catalog.lakeview_ml.worklist_scored w
  JOIN simo_show_case_catalog.lakeview_health.gold_index_admissions a USING (encounter_id)
  JOIN simo_show_case_catalog.lakeview_health.gold_discharge_features f USING (encounter_id)
  JOIN simo_show_case_catalog.lakeview_health.silver_patients p ON p.patient_id = a.patient_id
)
SELECT *,
  CASE WHEN risk_tier IN ('High', 'Medium') THEN ai_query('databricks-claude-haiku-4-5', concat(
    'You are a transitions-of-care nurse writing a brief for a care coordinator who will call this patient after discharge. ',
    'In at most 60 words write: one sentence on why this patient is at risk of 30-day readmission, then exactly 2 concrete actions for the call. ',
    'Use only the facts below. No greeting, no patient name, plain text.\n',
    'Age ', age_at_admit, ' ', sex, '; admitted for ', primary_condition, '; length of stay ', los_days, ' days; discharged to ', lower(discharge_disposition), '; ',
    num_discharge_meds, ' discharge medications; follow-up appointment scheduled: ', CASE WHEN followup_scheduled_7d THEN concat('yes (', followup_provider_type, ')') ELSE 'no' END, '; ',
    'risk percentile ', risk_percentile, '; top model risk drivers: ', top_drivers, '; social history: ', coalesce(social_risk_summary, 'none documented'), '.'))
  ELSE 'Low risk: standard discharge follow-up call per protocol.' END AS coordinator_brief,
  current_timestamp() AS scored_at
FROM base;

-- 1) App feed: source of the Lakebase synced table. UC does not allow row filters/column masks on synced-table sources,
--    so this table is restricted by GRANTs (app service principal + care-team admins only) and the app enforces facility scope.
CREATE TABLE IF NOT EXISTS simo_show_case_catalog.lakeview_ml.worklist_app_feed
COMMENT 'Restricted app feed (PHI): source of the Lakebase synced worklist. Same rows as coordinator_worklist. Access via GRANT only; facility scoping enforced in the app from lakeview_ml.facility_access.'
TBLPROPERTIES (delta.enableChangeDataFeed = true)
AS SELECT * FROM worklist_today WHERE false;
INSERT OVERWRITE simo_show_case_catalog.lakeview_ml.worklist_app_feed SELECT * FROM worklist_today;

-- 2) Governed worklist for analysts, Genie and SQL users: row filter by facility + PHI column masks (notebook 04).
--    INSERT OVERWRITE keeps the policies attached across daily runs.
CREATE TABLE IF NOT EXISTS simo_show_case_catalog.lakeview_ml.coordinator_worklist
COMMENT 'Daily transitions-of-care worklist: discharges still inside the 30-day readmission window, ranked by model risk, with top drivers and a GenAI coordinator brief. Governed: row filter by facility, masked PHI.'
AS SELECT * FROM simo_show_case_catalog.lakeview_ml.worklist_app_feed WHERE false;
INSERT OVERWRITE simo_show_case_catalog.lakeview_ml.coordinator_worklist SELECT * FROM simo_show_case_catalog.lakeview_ml.worklist_app_feed;

num_affected_rows,num_inserted_rows
254,254


In [0]:
%sql
SELECT risk_tier, count(*) AS patients, round(min(risk_percentile), 1) AS min_pct, round(max(risk_percentile), 1) AS max_pct,
       count_if(coordinator_brief IS NULL OR length(coordinator_brief) < 40) AS missing_briefs,
       round(avg(lace_score), 1) AS avg_lace
FROM simo_show_case_catalog.lakeview_ml.worklist_app_feed GROUP BY risk_tier ORDER BY max_pct DESC

risk_tier,patients,min_pct,max_pct,missing_briefs,avg_lace
High,71,80.3,99.9,0,13.0
Medium,60,60.2,79.5,0,10.5
Low,123,0.2,59.6,0,8.1


In [0]:
%sql
SELECT facility_id, primary_condition, days_since_discharge, risk_percentile, lace_score, top_drivers, coordinator_brief
FROM simo_show_case_catalog.lakeview_ml.coordinator_worklist
WHERE days_since_discharge <= 7
ORDER BY risk_percentile DESC LIMIT 5

facility_id,primary_condition,days_since_discharge,risk_percentile,lace_score,top_drivers,coordinator_brief
LVH-SOUTH,Heart failure,3,99.8,17,Frequent ED visits (6 mo) | Multiple comorbidities | Long length of stay,"Patient at high risk for readmission due to multiple comorbidities, recent frequent ED utilization, and extended hospitalization requiring close monitoring. Action 1: Schedule cardiology follow-up appointment within 7 days and confirm patient understanding of medication regimen, particularly diuretics and beta-blockers. Action 2: Arrange home health nursing visits for medication management and symptom monitoring after caregiver support ends in week two."
LVH-CENTRAL,Diabetes with complications,4,99.4,17,Frequent ED visits (6 mo) | Multiple comorbidities | Lives alone,"Patient is at high readmission risk due to advanced age, multiple comorbidities, frequent recent ED utilization, and lack of outpatient follow-up planning. Action 1: Schedule endocrinology or primary care appointment within 7 days and confirm attendance. Action 2: Assess alcohol use and connect patient with substance use counseling or support resources given daily consumption level."
LVH-CENTRAL,GI bleed,4,99.2,16,Multiple comorbidities | Frequent ED visits (6 mo) | Polypharmacy,"Patient at high risk due to multiple comorbidities, polypharmacy (13 medications), and recent frequent ED utilization. Action 1: Confirm understanding of discharge medications and identify potential drug interactions or adherence barriers. Action 2: Schedule follow-up appointment with primary care or gastroenterology within 7 days."
LVH-NORTH,COPD exacerbation,1,98.7,14,Frequent ED visits (6 mo) | Multiple comorbidities | No follow-up appointment scheduled,"Patient at high risk due to frequent ED visits, multiple comorbidities, and lack of scheduled follow-up appointment. Action 1: Confirm follow-up appointment with pulmonology within 7-10 days; assist with transportation resources if needed. Action 2: Review medication adherence barriers, explore patient assistance programs for cost, and assess alcohol use impact on COPD management."
LVH-SOUTH,Heart failure,1,98.5,11,Frequent ED visits (6 mo) | Lives alone | Abnormal labs at discharge,"Patient at high risk for readmission due to living alone without caregiver support, active smoking, and abnormal labs at discharge requiring close monitoring. Action 1: Confirm understanding of all 9 discharge medications, including dosing and side effects; assess barriers to adherence. Action 2: Schedule cardiology follow-up appointment and arrange home health nursing visits to monitor vital signs and labs."
